In [2]:
import sys
sys.path.append("../..")

from components.nl_sampler import NL_function_generator
from components.lagged_effects import LaggedEffects

import numpy as np
import matplotlib.pyplot as plt

In [5]:
# Example usage of using the structural lagged causal process  
# Note, we need an nl_sampler in the case that we want to sample nonlinear relationships

NL = NL_function_generator()

SE = LaggedEffects(
    n_vars=3,
    max_lags=2,
    link_proba=0.15,
    param_range=[0.3, 0.5],
    mirror_range=False,
    nonlinear_proba=0.5,
    nonstationary_change=0.1,
    stability_tolerance=100,   # Tolerance before resampling is stopped when the var process is unstable
    alternative_coeff_ts=0, 
    alternativ_parameter_range=[0.3, 0.5],
    alternative_link_proba=0.15,
    nl_sampler= NL
)



In [6]:
# The main functionality of the Structural Equation is "init_random_process" which initializes a random SCM according to the specified parameters.
# "get_step" can then be used to get the next Time Step of the process given the current Ts which has to be provided.

SE.init_random_process()
time_series = np.random.uniform(0,1, (3,100))
t = SE.get_step(time_series)


In [8]:
# If the equation can be set to be empty be setting link proba to 0 or specifying empty struc.
# we use this parameter to occasionally turn of the scm when generating data


SE = LaggedEffects(
    link_proba=0.0,
    nonlinear_proba=0

)

SE.init_random_process()
time_series = np.random.uniform(0,1, (3,100))
t = SE.get_step(time_series)
print(t)



SE = LaggedEffects(
    link_proba=0.3,
    nonlinear_proba=0

)

SE.init_random_process(empty_struc=True)
print(SE.links)

[[0.]
 [0.]
 [0.]]
[[[0. 0.]
  [0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]
  [0. 0.]]]


In [10]:
# If no nonlinearity is involved, the process can be initialized without the NL_sampler.


SE = LaggedEffects(
    nonlinear_proba=0.0,
)

SE.init_random_process()
time_series = np.random.uniform(0,1, (3,100))
t = SE.get_step(time_series)
print(t)

[[0.        ]
 [0.21566045]
 [0.        ]]


In [21]:
# Masks can be provided to force specific links and specific nl_relationships.

SE = LaggedEffects(
    link_proba=0.0,
    nonlinear_proba=0.0
)

# Bool Mask example:
mask= np.zeros((3,3,2)).astype(bool)
mask[0,1,0] = True

for x in range(3):
    SE.init_random_process(link_mask=mask)
    print(SE.links[0][1][0])
    
    

# Float Mask example:
mask= np.zeros((3,3,2)).astype(float)
mask[0,1,0] = 0.3

for x in range(3):
    SE.init_random_process(link_mask=mask)
    print(SE.links[0][1][0])
    
    
NL = NL_function_generator()

SE = LaggedEffects(
    link_proba=0.2,
    nonlinear_proba=0.2,
    nl_sampler= NL

)
    
# NL Mask example: (Note float masking for nl is not supported as it 
# is ambiguous given the nature of the distributions over NL functions.)
# To enforce the link to exist, the link_mask has to be set to True.

nl_mask= np.zeros((3,3,2)).astype(bool)
nl_mask[0,1,0] = True
for x in range(3):
    SE.init_random_process(nl_mask=nl_mask, link_mask=mask)
    print(SE.nl_naming[0][1][0])

0.4717195839822765
0.3453818698101768
0.3116605483378132
0.3
0.3
0.3
1.9226909349050882
1.9969176377347724
0.5826291924194358


In [23]:
# Masks can be provided to force specific links and specific nl_relationships.

SE = LaggedEffects(
    link_proba=0.0,
    nonlinear_proba=0.0
)

# Mask example:
mask= np.zeros((3,3,2)).astype(bool)
mask[0,1,0] = True


for x in range(3):
    SE.init_random_process(link_mask=mask)
    print(SE.links)

[[[0.         0.        ]
  [0.47171958 0.        ]
  [0.         0.        ]]

 [[0.         0.        ]
  [0.         0.        ]
  [0.         0.        ]]

 [[0.         0.        ]
  [0.         0.        ]
  [0.         0.        ]]]
[[[0.         0.        ]
  [0.34538187 0.        ]
  [0.         0.        ]]

 [[0.         0.        ]
  [0.         0.        ]
  [0.         0.        ]]

 [[0.         0.        ]
  [0.         0.        ]
  [0.         0.        ]]]
[[[0.         0.        ]
  [0.31166055 0.        ]
  [0.         0.        ]]

 [[0.         0.        ]
  [0.         0.        ]
  [0.         0.        ]]

 [[0.         0.        ]
  [0.         0.        ]
  [0.         0.        ]]]


In [27]:
# Links can additionally be restricted to a previous link tensor. This is only used if nonstationary change is > 0. 
# We use this feature when generating data that has change points in it.

SE = LaggedEffects(
    link_proba=0.3,
    nonlinear_proba=0.0,
    nonstationary_change=0.05
)

# Mask example:
previous_links= np.zeros((3,3,2)).astype(float)
previous_links[0,1,0] = 0.5


for x in range(3):
    SE.init_random_process(mask_restriction=previous_links)
    print(SE.links[0,1,0])

0.46414020800886174
0.47419122599146263
0.4726909349050884


In [34]:
# If necessary, the n last variable can be initialized to cause variables in the system with an alternative parameter range. 
# In out experiments, we use this feature to increase the confounding effects in the system (we remove the last variable.)


SE = LaggedEffects(
    link_proba=0.3,
    nonlinear_proba=0.0,
    alternative_coeff_ts = 1,
    alternativ_parameter_range = [0.8, 0.9],
    alternative_link_proba = 0.5
)


SE.init_random_process()
print(SE.links[:,:2])
print("__________")
print(SE.links[:,-1])

Attempt 1: The VAR process is not stable. Reinitializing with new random links.
Attempt 2: The VAR process is not stable. Reinitializing with new random links.
Attempt 3: The VAR process is not stable. Reinitializing with new random links.
Attempt 4: The VAR process is not stable. Reinitializing with new random links.
Attempt 5: The VAR process is not stable. Reinitializing with new random links.
Attempt 6: The VAR process is not stable. Reinitializing with new random links.
[[[0.         0.        ]
  [0.         0.        ]]

 [[0.36018974 0.39771681]
  [0.         0.        ]]

 [[0.35728925 0.        ]
  [0.         0.        ]]]
__________
[[0.         0.81058974]
 [0.81403396 0.84191143]
 [0.         0.        ]]


In [35]:
# A var stability test is run on default and links are resampled if it fails. Can be turned off if necessary.

SE = LaggedEffects(
    link_proba=1,
    nonlinear_proba=0.0,
    test_for_var_stability=False
)

SE.init_random_process()